# Pre-Requisites

## Load 'employee.csv' into DataFrame


In [0]:
df = spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.limit(4).display()

# Transaction 01 - Write data into Delta Format

## Logs are stored in .json format for each and every transaction or operation

In [0]:
df.write.format("delta").save(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta",
    mode="overwrite",
)

In [0]:
df_delta = spark.read.load(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta"
)
df_delta.limit(6).display()

## Create a View


In [0]:
df_delta.createOrReplaceTempView("user_vw")

In [0]:
%sql
UPDATE user_vw
SET country = 'IN'
WHERE country = 'India'

# Updating Delta Format file

In [0]:
df_delta = spark.read.load(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta"
)
df_delta.limit(6).display()

# Appending Delta Format file
- Used to overcome the data quality issue
- Do not allow append when there is a difference in schema.

In [0]:
spark.read.csv(
    path = "/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_006_new_column_education.csv",
    header = True,
    inferSchema = True
).write.format("delta").mode("append").save(
    path = "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta"
)

# Overwrite Delta Format file
- Old file is still present in the _delta_log

In [0]:
from pyspark.sql.functions import col
df.filter(col("country")=="India").write.format("delta").save(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta",
    mode="overwrite",
)

df_delta = spark.read.load(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta"
)
df_delta.limit(6).display()

# Transaction 02 - Read Delta Format file

In [0]:
from pyspark.sql.functions import col
df.filter(col("country")=="India").write.format("delta").save(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta",
    mode="overwrite",
)

In [0]:
df_delta = spark.read.load(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta"
)
df_delta.limit(10).display()

## Read old result (Maintains version)

In [0]:
df_delta = spark.read.load(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta",
    versionAsOf = 0
)
df_delta.limit(10).display()

# Read Transaction Log
- Each transaction is stored as a .json file.
- By reading the transaction we come to know the changes made, the name of parquet file created for the same, columns affected by the change etc...

## Approach 01 to read the transaction

In [0]:
spark.read.text(
  "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta/_delta_log/00000000000000000000.json"
).display()

## Approach 02 to read the transaction
- Using the delta.tables library
- Returns the table with all the transactions

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta/")
deltaTable.history().display()